# Description: Generation and evaluation of physio regressors contribution to the GS

This notebook will do the following:

1) Attempt to run AFNI program ```physio_calc``` on scans for which physiological timeseries are available.
2) Automaticall detect a subset of scans where ```physio_calc``` has done its job correctly
3) Compute the variance explained in the global signal by physiological regressors
4) Create a null distribution of variance explained

In [1]:
import os.path as osp
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import datetime
from utils.basics import PRJ_DIR, PRCS_DATA_DIR, SPRENG_DOWNLOAD_DIR, CODE_DIR, read_group_physio_reports
from sklearn.ensemble import IsolationForest
import hvplot.pandas
import holoviews as hv
from scipy.stats import zscore
import statsmodels.api as sm
from afnipy.lib_afni1D import Afni1D
import shutil

In [2]:
import getpass
username = getpass.getuser()
print(username)

javiergc


Get list of fMRI scans from the Spreng Dataset that passed our intial QC and are included in this study

In [3]:
dataset_info_df = pd.read_csv(osp.join(PRJ_DIR,'resources','good_scans.txt'))
dataset_info_df = dataset_info_df.set_index(['Subject','Session'])
print('++ Number of scans: %s scans' % dataset_info_df.shape[0])

++ Number of scans: 439 scans


***
# 1. Run ```physio_calc``` in all scans with physio data available

We will do this in biowulf using batch jobs. Here we create the necessary infrastructure for that

### 1.1. Create Swarm path

In [4]:
script_path = osp.join(PRJ_DIR,f'swarm.{username}',f'N09a_Compute_Physio_Regressors.SWARM.sh')
print(script_path)

/data/SFIMJGC_HCP7T/BCBL2024/swarm.javiergc/N09a_Compute_Physio_Regressors.SWARM.sh


### 1.2. Create folder for log files

In [5]:
log_path = osp.join(PRJ_DIR,f'logs.{username}',f'N09a_Compute_Physio_Regressors.log')
if not osp.exists(log_path):
    os.makedirs(log_path)
print(log_path)

/data/SFIMJGC_HCP7T/BCBL2024/logs.javiergc/N09a_Compute_Physio_Regressors.log


### 1.3. Write Swarm File

This file will contain one line per scan (sbj,ses) for which we were able to find both ```{sbj}_{ses}_task-rest_physio.tsv.gz``` and ```{sbj}_{ses}_task-rest_physio.json``` files

In [8]:
no_physio_scans = []

with open(script_path, 'w') as the_file:
    the_file.write('# Script Creation Date: %s\n' % str(datetime.date.today()))
    the_file.write(f'# swarm -f {script_path} -g 8 -t 8 -b 20 --time 00:10:00 --logdir {log_path} --partition quick,norm --module afni \n')
    the_file.write('\n')
    for sbj,ses in tqdm(dataset_info_df.index):
        physio_path = osp.join(SPRENG_DOWNLOAD_DIR,sbj,ses,'func',f'{sbj}_{ses}_task-rest_physio.tsv.gz')
        json_path   = osp.join(SPRENG_DOWNLOAD_DIR,sbj,ses,'func',f'{sbj}_{ses}_task-rest_physio.json')
        if osp.exists(physio_path) and osp.exists(json_path):
            for ec in [1,2,3]:
                dset_path = osp.join(SPRENG_DOWNLOAD_DIR,sbj,ses,'func',f'{sbj}_{ses}_task-rest_echo-{ec}_bold.nii.gz')
                out_dir = osp.join(PRCS_DATA_DIR,sbj,'D06_Physio')
                prefix = f'{sbj}_{ses}_task-rest_echo-{ec}'
                #the_file.write(f'physio_calc.py -phys_file {physio_path} -phys_json {json_path} -dset_epi {dset_path} -out_dir {out_dir} -prefix {prefix} \n')
                the_file.write(f'physio_calc.py -phys_file {physio_path} -phys_json {json_path} -dset_epi {dset_path} -out_dir {out_dir} -prefix {prefix} -prefilt_mode median -prefilt_max_freq 50\n')
        else:
            no_physio_scans.append((sbj,ses))    
the_file.close()     

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 439/439 [00:00<00:00, 682.49it/s]


### 1.4 Run batch jobs in biowulf

```bash

swarm -f /data/SFIMJGC_HCP7T/BCBL2024/swarm.javiergc/N09a_Compute_Physio_Regressors.SWARM.sh -g 8 -t 8 -b 20 --time 00:10:00 --logdir /data/SFIMJGC_HCP7T/BCBL2024/logs.javiergc/N09a_Compute_Physio_Regressors.log --partition quick,norm --module afni
```

***
# 2. Check the outputs of ```phys_calc```

For some scans, even though the physio files are available, they do not contain a sufficient amount of samples. In those cases ```physio_calc``` cannot run.

We next try to identify those instances and come with a final list of scans for which ```physio_calc``` completed. 

### 2.1. Get list of scans with existing automatically computed physiological regressors

First, we get the list of scans in which we attempted to run physio_calc

In [9]:
sbj_ses_with_physio = dataset_info_df.drop(no_physio_scans).index

Second, we check for which of these scans there is an ```slibase``` file. This is the primary output of ```phys_calc``` that contains the RVT and RETROICOR regressors.

In [12]:
sbj_ses_physio_corrupted = []
for sbj,ses in tqdm(sbj_ses_with_physio):
    for ec in [1,2,3]:
        file_path = osp.join(PRCS_DATA_DIR,sbj,'D06_Physio',f'{sbj}_{ses}_task-rest_echo-{ec}_slibase.1D')
        if not osp.exists(file_path):
            sbj_ses_physio_corrupted.append((sbj,ses))

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 426/426 [00:01<00:00, 259.94it/s]


In [13]:
sbj_ses_physio_corrupted = list(set(sbj_ses_physio_corrupted))
print('++ INFO: Number of scans with physio available, but somehow corrupted: %d scans' % len(sbj_ses_physio_corrupted))
print(sbj_ses_physio_corrupted)

++ INFO: Number of scans with physio available, but somehow corrupted: 56 scans
[('sub-15', 'ses-2'), ('sub-118', 'ses-2'), ('sub-29', 'ses-2'), ('sub-43', 'ses-2'), ('sub-74', 'ses-1'), ('sub-106', 'ses-1'), ('sub-32', 'ses-1'), ('sub-30', 'ses-1'), ('sub-37', 'ses-2'), ('sub-39', 'ses-2'), ('sub-45', 'ses-2'), ('sub-23', 'ses-2'), ('sub-34', 'ses-2'), ('sub-36', 'ses-1'), ('sub-51', 'ses-1'), ('sub-55', 'ses-2'), ('sub-19', 'ses-2'), ('sub-35', 'ses-2'), ('sub-47', 'ses-2'), ('sub-127', 'ses-2'), ('sub-22', 'ses-1'), ('sub-29', 'ses-1'), ('sub-43', 'ses-1'), ('sub-15', 'ses-1'), ('sub-18', 'ses-1'), ('sub-24', 'ses-2'), ('sub-41', 'ses-1'), ('sub-129', 'ses-1'), ('sub-219', 'ses-2'), ('sub-76', 'ses-2'), ('sub-27', 'ses-1'), ('sub-45', 'ses-1'), ('sub-38', 'ses-1'), ('sub-25', 'ses-2'), ('sub-128', 'ses-1'), ('sub-28', 'ses-2'), ('sub-37', 'ses-1'), ('sub-55', 'ses-1'), ('sub-16', 'ses-2'), ('sub-21', 'ses-2'), ('sub-17', 'ses-2'), ('sub-47', 'ses-1'), ('sub-127', 'ses-1'), ('sub-30'

Finally, we create a new list that only contains the scans for which ```physio_calc``` was able to generate a ```slibase``` file.

In [14]:
scans_with_complete_physio = dataset_info_df.drop(no_physio_scans + sbj_ses_physio_corrupted).index
print("++ INFO: Number of scans for which we were able to complete physio_calc = %d scans" % len(scans_with_complete_physio))

++ INFO: Number of scans for which we were able to complete physio_calc = 370 scans


## 2.2 Check some basic statistics to ensure consistent physiological regressors quality

First, we will use AFNI's ```gen_ss_review_table.py``` to gather into a single file quality metrics regarding peak detection on the cardiac and respiratory traces. As one can expect for cardiac and respiratory rates to be somehow within a limited range, looking for scans where these are outliers is one good way to automatically detect scans with incorrect physiological regressors.

```bash
ml afni
cd /data/SFIMJGC_HCP7T/BCBL2024/prcs_data

gen_ss_review_table.py -overwrite \
                       -write_table ./physio_card_review_all_scans.txt \
                       -infiles ./sub-*/D06_Physio/sub-*_ses-?_task-rest_echo-1_card_review.txt

gen_ss_review_table.py -overwrite \
                       -write_table ./physio_resp_review_all_scans.txt \
                       -infiles ./sub-*/D06_Physio/sub-*_ses-?_task-rest_echo-1_resp_review.txt
                       
```

We now read these pysio report files, one for cardiac regressors and one for respiration regressors

In [15]:
report_card_summary_path  = osp.join(PRJ_DIR,'prcs_data','physio_card_review_all_scans.txt')
report_resp_summary_path  = osp.join(PRJ_DIR,'prcs_data','physio_resp_review_all_scans.txt')

### 2.3 Detect scans where peak detection for cardiac traces might have failed

To detect potential errors in cardiac peak detection, we will characterize each scan by its mean inter-peak interval and its standard devation. We will then use IsolationForest to detect outliers, namely scans whose mean and standard deviation deviate from the most common ones in the sample

In [16]:
report_card_summary_df = read_group_physio_reports(report_card_summary_path)

In [17]:
clf    = IsolationForest(contamination=0.1, random_state=42)
labels = clf.fit_predict(report_card_summary_df['peak ival over dset mean std'])
outliers = labels == -1
df_card = report_card_summary_df['peak ival over dset mean std'].copy()
df_card.columns=['Mean','St.Dev.']
df_card['color'] = ['red' if c else 'green' for c in outliers]
df_card.hvplot.scatter(x='Mean',y='St.Dev.', c='color', title='Cardiac Inter-peak Interval (seconds)', aspect='square', hover_cols=['Subject','Run'], alpha=0.5) * hv.VSpan(60,100)

:Overlay
   .Scatter.I :Scatter   [Mean]   (St.Dev.,color,Subject,Run)
   .VSpan.I   :VSpan   [x,y]

In the figure above, each dot represents a scan. Red dots are scans marked as outliers.

Finally, we create a list with the non-outlier scans. These are scans that, from the perspective of peak detection in the cardiac traces, are valid for further analyses.

In [18]:
scans_with_reasonable_cardiac = df_card[df_card['color']=='green'].index
len(scans_with_reasonable_cardiac)

333

### 2.4 Detect scans where peak detection for respiratory traces might have failed

Here we apply the same logic as in the above section, but this time looking at the inter-peak interval statistics for the respiratory traces

In [19]:
report_resp_summary_df = read_group_physio_reports(report_resp_summary_path)

In [20]:
clf    = IsolationForest(contamination=0.1, random_state=42)
labels = clf.fit_predict(report_resp_summary_df['peak ival over dset mean std'])
outliers = labels == -1
df_resp = report_resp_summary_df['peak ival over dset mean std'].copy()
df_resp.columns=['Mean','St.Dev.']
df_resp['color'] = ['red' if c else 'green' for c in outliers]
df_resp.hvplot.scatter(x='Mean',y='St.Dev.', c='color', title='Cardiac Inter-peak Interval (seconds)', aspect='square', hover_cols=['Subject','Run'], alpha=0.5) * hv.VSpan(60,100)

:Overlay
   .Scatter.I :Scatter   [Mean]   (St.Dev.,color,Subject,Run)
   .VSpan.I   :VSpan   [x,y]

In the figure above, each dot represents a scan. Red dots are scans marked as outliers.

Finally, we create a list with the non-outlier scans. These are scans that, from the perspective of peak detection in the respiratory traces, are valid for further analyses.

In [21]:
scans_with_reasonable_resp = df_resp[df_resp['color']=='green'].index
len(scans_with_reasonable_resp)

333

### 2.5. Combine information gathered from cardiac and respiratory peak detection to create a final list of scans

We now create a final list of scans with valid physiological data by keeping only scans not marked as outliers both from the cardiac and respiration perspective.

In [22]:
selected_scans = scans_with_reasonable_cardiac.intersection(scans_with_reasonable_resp)
print("++ INFO: Number of scans with reasonable physiological regressors: %d scans" % len(selected_scans))

++ INFO: Number of scans with reasonable physiological regressors: 303 scans


***

# 3. Compute variance explained in the GS for physiological regressors

For the scans that we know have good physio regressors, we will now compute how much variance of the global signal can be explained by the physio regressors. This is done via batch jobs that call program ```GS_physio_exp_var```. The way this program estimates variance explained is as follows:

1. Load provided global signal and physiological regressors into memory
2. Remove constant and linear trends from all loaded timeseries separately
3. For each regressor type (e.g., rvt01, rtv02, card.c1, etc) it finds the time-shifted version that most strongly correlates with the global signal. At the end of this step, we will have a list of 13 regressors.
4. Computes the variance explained by these 13 regressors.

### 3.1. Create path for Swarm file

In [23]:
script_path = osp.join(PRJ_DIR,f'swarm.{username}',f'N09b_Compute_varexp_in_GS_by_physio.SWARM.sh')
print(script_path)

/data/SFIMJGC_HCP7T/BCBL2024/swarm.javiergc/N09b_Compute_varexp_in_GS_by_physio.SWARM.sh


### 3.2. Create folder for logs

In [24]:
log_path = osp.join(PRJ_DIR,f'logs.{username}',f'N09b_Compute_varexp_in_GS_by_physio.log')
if not osp.exists(log_path):
    os.makedirs(log_path)
print(log_path)

/data/SFIMJGC_HCP7T/BCBL2024/logs.javiergc/N09b_Compute_varexp_in_GS_by_physio.log


### 3.3. Write Swarm file

This will contain one line per-scan that we have marked as having good physiological data.

In [27]:
with open(script_path, 'w') as the_file:
    the_file.write('# Script Creation Date: %s\n' % str(datetime.date.today()))
    the_file.write(f'# swarm -f {script_path} -g 8 -t 8 -b 20 --time 00:10:00 --logdir {log_path} --partition quick,norm --module afni \n')
    the_file.write('\n')
    for sbj,ses in tqdm(selected_scans):
        gs_path      = osp.join(PRCS_DATA_DIR,sbj,f'D02_Preproc_fMRI_{ses}',f'pb03.{sbj}.r01.e02.volreg.scale.GSasis.1D')
        slibase_path = osp.join(PRCS_DATA_DIR,sbj,'D06_Physio',f'{sbj}_{ses}_task-rest_echo-2_slibase.1D')
        output_path  = osp.join(PRCS_DATA_DIR,sbj,f'D02_Preproc_fMRI_{ses}',f'pb03.{sbj}.r01.e02.volreg.scale.GSasis.PhysioModeling.pkl')
        the_file.write(f'export GS_PATH={gs_path} PHYSIO_PATH={slibase_path} OUTPUT_PATH={output_path}; sh {CODE_DIR}/python/GS_physio_exp_var.sh\n')
the_file.close()     

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 303/303 [00:00<00:00, 163141.73it/s]


The next cell help us look for issues when running the batch jobs. If all things went well there should be no WARNING lines printed out.

In [28]:
for sbj,ses in tqdm(selected_scans):
    output_path  = osp.join(PRCS_DATA_DIR,sbj,f'D02_Preproc_fMRI_{ses}',f'pb03.{sbj}.r01.e02.volreg.scale.GSasis.PhysioModeling.pkl')
    if not osp.exists(output_path):
        print("++ WARNING: %s is missing" % output_path)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 303/303 [00:00<00:00, 409.09it/s]


We will now compile all results into a single csv file for later exploration

In [29]:
df = pd.DataFrame(index=selected_scans,columns=['Var. Exp. by Physio Regressors'])
for sbj,ses in tqdm(selected_scans):
    # Variance Explained by Regressors
    model_path  = osp.join(PRCS_DATA_DIR,sbj,f'D02_Preproc_fMRI_{ses}',f'pb03.{sbj}.r01.e02.volreg.scale.GSasis.PhysioModeling.pkl')
    model = sm.load(model_path)
    df.loc[(sbj,ses),'Var. Exp. by Physio Regressors'] = model.rsquared
df=df.infer_objects()
df.to_csv('./cache/real_varexp_gs_physio.csv')

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 303/303 [00:01<00:00, 289.85it/s]


***

# 4. Create a NULL DISTRIBUTION for estimates of variance explained.

Both the global signal and the physiological regressors have quite constrained spectral characteristics. Moreover, to ensure we do not understimate how much physiology can explain the global signal, we are picking the best time-shited version of each regressor. Although those are good things to make sure we do not understimate the contribution of cardiac and respiratory function to the global signal, it can lead to over estimation. By generating a null distribution were we compute the variance explain in the global signal of one scan by the physiological regressors of another scan, we build a null distribution so that we can better contextualize our variance explained estimates.

### 4.1. Create path for swarm file

In [30]:
script_path = osp.join(PRJ_DIR,f'swarm.{username}',f'N09c_Compute_varexp_in_GS_by_physio_nulls.SWARM.sh')
print(script_path)

/data/SFIMJGC_HCP7T/BCBL2024/swarm.javiergc/N09c_Compute_varexp_in_GS_by_physio_nulls.SWARM.sh


### 4.2. Create folder for logs

In [31]:
log_path = osp.join(PRJ_DIR,f'logs.{username}',f'N09c_Compute_varexp_in_GS_by_physio_nulls.log')
if not osp.exists(log_path):
    os.makedirs(log_path)
print(log_path)

/data/SFIMJGC_HCP7T/BCBL2024/logs.javiergc/N09c_Compute_varexp_in_GS_by_physio_nulls.log


### 4.3. Create a folder where to save the results of each of the 10,000 null permutations

In [32]:
perm_dir = osp.join(CODE_DIR,'notebooks','cache','gs_phys_varex_perms')
if osp.exists(perm_dir):
    shutil.rmtree(perm_dir)
os.makedirs(perm_dir)

### 4.4. Write the Swarm file

Here, for each permutation, we first randomly select one scan (sbj,ses) for the global signal. Then we randomly select one scan from any other subject for the physiological regressors.

In [33]:
n_null_cases = 10000
selected_scans_df = pd.DataFrame(index=selected_scans)
with open(script_path, 'w') as the_file:
    the_file.write('# Script Creation Date: %s\n' % str(datetime.date.today()))
    the_file.write(f'# swarm -f {script_path} -g 8 -t 8 -b 20 --time 00:10:00 --logdir {log_path} --partition quick,norm --module afni \n')
    the_file.write('\n')
    for i in tqdm(range(n_null_cases)):
        ii = str(i).zfill(5)
        gs_sbj, gs_ses = selected_scans_df.sample(1).index.values[0]
        ph_sbj, ph_ses = pd.DataFrame(index=selected_scans.drop(gs_sbj,level='Subject')).sample(1).index.values[0]
        gs_path        = osp.join(PRCS_DATA_DIR,gs_sbj,f'D02_Preproc_fMRI_{gs_ses}',f'pb03.{gs_sbj}.r01.e02.volreg.scale.GSasis.1D')
        ph_path        = osp.join(PRCS_DATA_DIR,ph_sbj,'D06_Physio',f'{ph_sbj}_{ph_ses}_task-rest_echo-2_slibase.1D')
        out_path       = osp.join(perm_dir,f'gs_phys_varex_{ii}.pkl')
        the_file.write(f'export GS_PATH={gs_path} PHYSIO_PATH={ph_path} OUTPUT_PATH={out_path}; sh {CODE_DIR}/python/GS_physio_exp_var.sh\n')
the_file.close()     

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:05<00:00, 1695.67it/s]


### 4.5. Check all permutations finished correctly

In [35]:
for i in tqdm(range(n_null_cases)):
    ii = str(i).zfill(5)
    output_path  = osp.join(perm_dir,f'gs_phys_varex_{ii}.pkl')
    if not osp.exists(output_path):
        print("++ WARNING: %s is missing" % output_path)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:06<00:00, 1489.04it/s]


We will now compile all results into a single csv file for later exploration

In [36]:
df = pd.DataFrame(index=range(n_null_cases),columns=['Var. Exp. by Physio Regressors (NULL)'])
for i in tqdm(range(n_null_cases)):
    ii = str(i).zfill(5)
    model_path  = osp.join(perm_dir,f'gs_phys_varex_{ii}.pkl')
    model = sm.load(model_path)
    df.loc[i,'Var. Exp. by Physio Regressors (NULL)'] = model.rsquared
df=df.infer_objects()
df.index.name='Permutation'
df.to_csv('./cache/null_varexp_gs_physio.csv')

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:19<00:00, 504.06it/s]


***

# CODE TO DELETE


# Check Results

In [142]:
df_null = pd.DataFrame(index=range(n_null_cases), columns=['Varexp. by Physio'])
for i in tqdm(range(n_null_cases)):
    ii = str(i).zfill(5)
    model_path  = osp.join(perm_dir,f'gs_phys_varex_{ii}.pkl')
    model = sm.load(model_path)
    df_null.loc[i,'Varexp. by Physio'] = model.rsquared
df_null = df_null.infer_objects()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:10<00:00, 955.40it/s]


In [143]:
df_null.hvplot.kde()

:Distribution   [Varexp. by Physio]   (Density)

In [144]:
df = pd.DataFrame(index=selected_scans,columns=['Varexp. by Physio','kappa','rho','Mean Motion (enorm)','Max Motion (enorm)'])
for sbj,ses in tqdm(selected_scans):
    # Variance Explained by Regressors
    model_path  = osp.join(PRCS_DATA_DIR,sbj,f'D02_Preproc_fMRI_{ses}',f'pb03.{sbj}.r01.e02.volreg.scale.GSasis.PhysioModeling.pkl')
    model = sm.load(model_path)
    df.loc[(sbj,ses),'Varexp. by Physio'] = model.rsquared
    # Kappa and Rho
    kr_path = osp.join(PRCS_DATA_DIR,sbj,f'D02_Preproc_fMRI_{ses}',f'{sbj}_{ses}_GS_kappa_and_rho.txt')
    kr      = pd.read_csv(kr_path)
    df.loc[(sbj,ses),'kappa'] = kr.loc[0,'kappa']
    df.loc[(sbj,ses),'rho'] = kr.loc[0,'rho']
    # Motion
    mot_path = osp.join(PRCS_DATA_DIR,sbj,f'D04_Preproc_fMRI_{ses}_NORDIC',f'motion_{sbj}_enorm.1D')
    aux_mot = np.loadtxt(mot_path)
    df.loc[(sbj,ses),'Mean Motion (enorm)'] = aux_mot.mean()
    df.loc[(sbj,ses),'Max Motion (enorm)'] = aux_mot.max()
df =df.infer_objects()

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 305/305 [00:01<00:00, 179.37it/s]


In [149]:
df_null.hvplot.kde(label='Null Distribution') * df['Varexp. by Physio'].hvplot.kde(label='Real Data')

:Overlay
   .Distribution.Null_Distribution :Distribution   [Varexp. by Physio]   (Density)
   .Distribution.Real_Data         :Distribution   [Varexp. by Physio]   (Density)

In [158]:
sample_sbj, sample_ses = df.sort_values(by='Varexp. by Physio', ascending=False).iloc[0].name

In [160]:
sample_gs_path = osp.join(PRCS_DATA_DIR,sample_sbj,f'D02_Preproc_fMRI_{sample_ses}',f'pb03.{sample_sbj}.r01.e02.volreg.scale.GSasis.1D')
sample_gs      = pd.DataFrame(np.loadtxt(sample_gs_path),columns=['orig'])
sample_gs.hvplot()

:Curve   [index]   (orig)

In [56]:
df.hvplot.scatter(x='Varexp. by Physio',y='kappa', aspect='square')

:Scatter   [Varexp. by Physio]   (kappa)

In [66]:
a = df_resp.loc[selected_scans]
a = a[['Mean','St.Dev.']]
a.columns = ['Resp Mean ipi','Resp St.Dev. ipi']
pd.concat([df,a],axis=1).hvplot.scatter(x='Varexp. by Physio',y='Resp St.Dev. ipi', aspect='square')

:Scatter   [Varexp. by Physio]   (Resp St.Dev. ipi)

In [63]:
df_resp

Mean   St.Dev.  color
Subject Run                              
sub-01  ses-1   3.122436  0.236670  green
        ses-2   3.012748  0.268144  green
sub-02  ses-1   4.211111  0.562136  green
        ses-2   6.983908  1.829205  green
sub-03  ses-1  11.856373  1.936529    red
...                  ...       ...    ...
sub-96  ses-2   3.027612  0.292531  green
sub-98  ses-1   4.041833  0.416368  green
        ses-2   4.119595  0.469253  green
sub-99  ses-1   3.502011  0.715958  green
        ses-2   3.473286  0.445779  green

[373 rows x 3 columns]

In [43]:
a.loc[0,'kappa']

np.float64(67.53014070025269)

***
# Computation of variance explained

In [162]:
osp.exists(osp.dirname("/data/SFIMJGC/ppp.j"))

True

In [44]:
import statsmodels.api as sm

def variance_explained(df_target, df_predictors):
    # Ensure inputs are compatible
    y = df_target.iloc[:, 0]  # assuming only one column in target
    X = df_predictors

    # Add intercept term to predictors
    X = sm.add_constant(X)

    # Fit OLS model
    model = sm.OLS(y, X).fit()

    # R-squared: proportion of variance in y explained by X
    return model.rsquared, model

In [22]:
def detrend_signal(y):
    nt    = len(y)
    trend = sm.add_constant(np.arange(nt))
    y_detrended = sm.OLS(y, trend).fit().resid
    return y_detrended
def variance_explained_after_trend(df_target, df_predictors):
    y = df_target.iloc[:, 0]
    X = df_predictors.copy()

    # Create linear trend regressor
    trend = sm.add_constant(np.arange(len(y)))

    # Regress out trend from y
    y_detrended = sm.OLS(y, trend).fit().resid

    # Regress out trend from each predictor
    X_detrended = X.apply(lambda col: sm.OLS(col, trend).fit().resid)

    # Add intercept to detrended predictors
    X_detrended = sm.add_constant(X_detrended)

    # Fit model on detrended data
    model = sm.OLS(y_detrended, X_detrended).fit()

    return model.rsquared, model

In [71]:
from utils.basics import detrend_signal

In [89]:
sbj_a,ses_a = selected_scans[10]
sbj_b,ses_b = selected_scans[100]

In [90]:
# Read and detrend global signal
gs_path = osp.join(PRCS_DATA_DIR,sbj_a,f'D02_Preproc_fMRI_{ses_a}',f'pb03.{sbj_a}.r01.e02.volreg.scale.GSasis.1D')
gs_df   = pd.DataFrame(np.loadtxt(gs_path))
gs_df.columns = ['orig']
gs_df['det'] = detrend_signal(gs_df['orig'].values.squeeze())

In [91]:
gs_df['det'].hvplot()

:Curve   [index]   (det)

In [92]:
slibase_path = osp.join(PRCS_DATA_DIR,sbj_b,'D06_Physio',f'{sbj_b}_{ses_b}_task-rest_echo-2_slibase.1D')
slibase_obj  = Afni1D(slibase_path)
slibase_df   = pd.read_csv(slibase_path, comment='#', delimiter=' +', header=None, engine='python')
slibase_df.columns=slibase_obj.labels
slibase_df=slibase_df[3::].reset_index(drop=True)
slibase_det_df = slibase_df.copy()
for c in slibase_det_df.columns:
    slibase_det_df[c] = detrend_signal(slibase_df[c].values.squeeze())
phys_reg_list = np.unique([c.split('.',1)[1] for c in slibase_det_df.columns])

In [93]:
corrs_with_physio = pd.concat([gs_df,slibase_det_df],axis=1).corr().loc['det']
sel_physio_regs = []
for pr in phys_reg_list:
    aux = (corrs_with_physio.loc[[i for i in corrs_with_physio.index if pr in i]].abs().sort_values(ascending=False)).index[0]
    sel_physio_regs.append(aux)
sel_physio_regs

['s010.card.c1',
 's041.card.c2',
 's018.card.s1',
 's045.card.s2',
 's041.resp.c1',
 's033.resp.c2',
 's013.resp.s1',
 's039.resp.s2',
 's000.rvt00',
 's000.rvt01',
 's000.rvt02',
 's000.rvt03',
 's000.rvt04']

In [94]:
y = gs_df['det']
X = slibase_det_df[sel_physio_regs]
X = sm.add_constant(X)
model = sm.OLS(y,X).fit()

In [95]:
model.rsquared

np.float64(0.11716978019663882)

In [155]:
gs_df['det_pred'] = model.predict()

In [156]:
gs_df[['det','det_pred']].apply(zscore).hvplot()

:NdOverlay   [Variable]
   :Curve   [index]   (value)

In [170]:
slibase_df[['s000.card.s1','s001.card.s1','s002.card.s1']].hvplot(width=1000)

:NdOverlay   [Variable]
   :Curve   [index]   (value)

In [113]:
model.rsquared

np.float64(0.1794398331218212)

In [202]:
slibase_path = osp.join(PRCS_DATA_DIR,sbj_b,'D06_Physio',f'{sbj_b}_{ses_b}_task-rest_echo-2_slibase.1D')
slibase_obj  = Afni1D(slibase_path)
slibase_df   = pd.read_csv(slibase_path, comment='#', delimiter=' +', header=None, engine='python')
slibase_df.columns=slibase_obj.labels
slibase_df=slibase_df[3::].reset_index(drop=True)

In [281]:
gs_df['det'] = detrend_signal(gs.values.squeeze())

In [282]:
gs_df['det'] = gs_det

In [284]:
gs['orig'].hvplot()

KeyError: 'orig'

In [264]:
r2, model = variance_explained(gs_df, slibase_df[[c for c in slibase_df.columns if ('s000.card.c1') in c ]])
print(f"Variance explained (R²): {r2:.3f}")

Variance explained (R²): 0.023


In [244]:
r2, model = variance_explained_after_trend(gs_df, slibase_df[[c for c in slibase_df.columns if ('10.card.c1' in c) ]])
print(f"Variance explained (R²): {r2:.3f}")

Variance explained (R²): 0.037


In [228]:
slibase_df[[c for c in slibase_df.columns if '2.card.c1' in c ]].apply(zscore).hvplot(width=2000) * gs_df.apply(zscore).hvplot(width=2000).opts(line_width=5,line_color='k') 

:Overlay
   .NdOverlay.I :NdOverlay   [Variable]
      :Curve   [index]   (value)
   .Curve.I     :Curve   [index]   (0)

In [271]:
from sklearn.metrics import explained_variance_score, r2_score

In [269]:
slibase_df['s000.card.c1'].values.squeeze().shape

(201,)

In [272]:
r2_score(y_true=gs_df.values.squeeze(),y_pred=slibase_df['s000.card.c1'].values.squeeze())

-89075.54156162347

In [254]:
np.corrcoef(gs_df.values.T,slibase_df['s000.card.c1'].values)

array([[1.        , 0.15077351],
       [0.15077351, 1.        ]])

In [245]:
model.summary2()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                 Results: Ordinary least squares
=================================================================
Model:              OLS              Adj. R-squared:     0.032   
Dependent Variable: y                AIC:                32.7325 
Date:               2025-06-17 07:14 BIC:                39.3391 
No. Observations:   201              Log-Likelihood:     -14.366 
Df Model:           1                F-statistic:        7.583   
Df Residuals:       199              Prob (F-statistic): 0.00644 
R-squared:          0.037            Scale:              0.068226
------------------------------------------------------------------
               Coef.   Std.Err.     t     P>|t|    [0.025   0.975]
------------------------------------------------------------------
const         -0.0000    0.0184  -0.0000  1.0000  -0.0363   0.0363
s010.card.c1  -0.0723    0.0262  -2.7537  0.0064  -0.1240  -0.0205
-----------------------------------------------------------------
Omnibus:              1.878        Durbin-Watson:           0.614
Prob(Omnibus):        0.391        Jarque-Bera (JB):        1.533
Skew:                 0.123        Prob(JB):                0.465
Kurtosis:             3.350        Condition No.:           1    
=================================================================
Notes:
[1] Standard Errors assume that the covariance matrix of the
errors is correctly specified.
"""

In [221]:
slibase_df.columns

Index(['s000.rvt00', 's000.rvt01', 's000.rvt02', 's000.rvt03', 's000.rvt04',
       's000.card.c1', 's000.card.s1', 's000.card.c2', 's000.card.s2',
       's000.resp.c1',
       ...
       's045.rvt03', 's045.rvt04', 's045.card.c1', 's045.card.s1',
       's045.card.c2', 's045.card.s2', 's045.resp.c1', 's045.resp.s1',
       's045.resp.c2', 's045.resp.s2'],
      dtype='object', length=598)

In [103]:
report_card_summary_df[('HR','Mean')]   = df[('peak ival over dset mean std','value_1')]
report_card_summary_df[('HR','St. Dev.')] = df[('peak ival over dset mean std','value_2')]

In [101]:
report_card_summary_df['peak num over dset'].hvplot.hist(bins=100, title='Distribution of number of cardiac peaks during a run', xlabel='Number of peaks',ylabel='# of Scans')

:Histogram   [value_1]   (Count)

In [102]:
df['HR'].hvplot.scatter(x='Mean',y='St. Dev.', xlabel='Mean',ylabel='St. Dev.', title='Mean HR (b.p.m.)') * hv.VSpan(60,100)

:Overlay
   .Scatter.I :Scatter   [Mean]   (St. Dev.)
   .VSpan.I   :VSpan   [x,y]

In [35]:
report_card_keepers_df = read_gen_ss_review_table(report_card_keepers_path)

++ INFO [read_gen_ss_review_table]: Number of scans = 42 | Number of metrics per scan = 6


In [36]:
report_card_keepers_df.sort_values(by='peak num over dset').head(5)

,group,subject,peak num over dset,read_in sampling freq,ts_orig end time,dset start time
26,198.0,sub-198_ses-2,602.0,40.0,612.15,0.0
22,182.0,sub-182_ses-2,602.0,40.0,612.15,0.0
25,198.0,sub-198_ses-1,603.0,40.0,612.15,0.0
40,73.0,sub-73_ses-1,604.0,40.0,612.15,0.0
33,48.0,sub-48_ses-2,605.0,40.0,612.15,0.0


In [57]:
df = pd.read_csv(report_card_summary_path, sep='\t', header=[0,1] )
original_columns  = [(a,b) for a,b in df.loc[[0,1]].values.T]
df.drop([0,1], inplace=True)
new_columns = []
df

,group,subject,filename,label,read_in sampling freq,prefilt mode,"prefilt window, sec",is user interact on?,num peaks changed,ts_orig sampling freq,...,Unnamed: 25_level_0,Unnamed: 26_level_0,peak num over dset,peak ival over dset min max,Unnamed: 29_level_0,peak ival over dset mean std,Unnamed: 31_level_0,peak ival over dset q25 q50 q75,Unnamed: 33_level_0,Unnamed: 34_level_0
,value,value,value_1,value_1,value_1,value_1,value_1,value_1,value_1,value_1,...,value_2,value_3,value_1,value_1,value_2,value_1,value_2,value_1,value_2,value_3
2,2,sub-02_ses-1,/data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...,card,40,none,NaN,False,0,40.0,...,1.225,1.225,505,1.150,1.275,1.212401,0.027413,1.200,1.225,1.225
3,2,sub-02_ses-2,/data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...,card,40,none,NaN,False,0,40.0,...,1.200,1.225,505,1.075,1.300,1.211508,0.032756,1.200,1.200,1.225
4,3,sub-03_ses-1,/data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...,card,40,none,NaN,False,0,40.0,...,1.050,1.075,582,0.425,1.375,1.051893,0.068631,1.025,1.050,1.075
5,3,sub-03_ses-2,/data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...,card,40,none,NaN,True,0,40.0,...,1.075,1.075,574,0.625,1.325,1.066318,0.034825,1.050,1.075,1.075
6,4,sub-04_ses-1,/data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...,card,40,none,NaN,False,0,40.0,...,1.075,1.075,574,0.825,1.725,1.066623,0.043284,1.050,1.075,1.075
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
368,96,sub-96_ses-2,/data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...,card,40,none,NaN,False,0,40.0,...,0.975,1.000,635,0.775,1.125,0.963880,0.060708,0.925,0.975,1.000
369,98,sub-98_ses-1,/data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...,card,40,none,NaN,False,0,40.0,...,1.400,1.450,437,1.100,1.625,1.400344,0.074211,1.350,1.400,1.450
370,98,sub-98_ses-2,/data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...,card,40,none,NaN,False,0,40.0,...,1.400,1.450,441,1.100,1.625,1.387557,0.079487,1.350,1.400,1.450


In [58]:
df = pd.read_csv(report_card_summary_path, sep='\t', header=[0,1] )
new_columns = []
current_lvl1 = None

for lvl1, lvl2 in df.columns:
    if "Unnamed" in str(lvl1):
        new_columns.append((current_lvl1, lvl2))
    else:
        current_lvl1 = lvl1
        new_columns.append((lvl1, lvl2))

df.columns = pd.MultiIndex.from_tuples(new_columns)

In [59]:
df

group       subject                                           filename  \
    value         value                                            value_1   
0       1  sub-01_ses-1  /data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...   
1       1  sub-01_ses-2  /data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...   
2       2  sub-02_ses-1  /data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...   
3       2  sub-02_ses-2  /data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...   
4       3  sub-03_ses-1  /data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...   
..    ...           ...                                                ...   
368    96  sub-96_ses-2  /data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...   
369    98  sub-98_ses-1  /data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...   
370    98  sub-98_ses-2  /data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...   
371    99  sub-99_ses-1  /data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...   
372    99  sub-99_ses-2  /data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...   

      label read_in sampling freq prefilt mode prefilt window, sec  \
    value_1               value_1      value_1             value_1   
0      card                    40         none                 NaN   
1      card                    40         none                 NaN   
2      card                    40         none                 NaN   
3      card                    40         none                 NaN   
4      card                    40         none                 NaN   
..      ...                   ...          ...                 ...   
368    card                    40         none                 NaN   
369    card                    40         none                 NaN   
370    card                    40         none                 NaN   
371    card                    40         none                 NaN   
372    card                    40         none                 NaN   

    is user interact on? num peaks changed ts_orig sampling freq  ...  \
                 value_1           value_1               value_1  ...   
0                  False                 0                  40.0  ...   
1                  False                 0                  40.0  ...   
2                  False                 0                  40.0  ...   
3                  False                 0                  40.0  ...   
4                  False                 0                  40.0  ...   
..                   ...               ...                   ...  ...   
368                False                 0                  40.0  ...   
369                False                 0                  40.0  ...   
370                False                 0                  40.0  ...   
371                False                 0                  40.0  ...   
372                False                 0                  40.0  ...   

    peak ival q25 q50 q75         peak num over dset  \
                  value_2 value_3            value_1   
0                   0.875   0.925                706   
1                   0.875   0.900                711   
2                   1.225   1.225                505   
3                   1.200   1.225                505   
4                   1.050   1.075                582   
..                    ...     ...                ...   
368                 0.975   1.000                635   
369                 1.400   1.450                437   
370                 1.400   1.450                441   
371                 0.750   0.775                814   
372                 0.775   0.800                789   

    peak ival over dset min max         peak ival over dset mean std  \
                        value_1 value_2                      value_1   
0                         0.650   1.000                     0.867057   
1                         0.700   1.000                     0.861092   
2                         1.150   1.275                     1.212401   
3                         1.075   1.300

In [55]:
original_columns

[('group', 'value'),
 ('subject', 'value'),
 ('filename', 'value_1'),
 ('label', 'value_1'),
 ('read_in sampling freq', 'value_1'),
 ('prefilt mode', 'value_1'),
 ('prefilt window, sec', 'value_1'),
 ('is user interact on?', 'value_1'),
 ('num peaks changed', 'value_1'),
 ('ts_orig sampling freq', 'value_1'),
 ('ts_orig sampling rate', 'value_1'),
 ('ts_orig num points', 'value_1'),
 ('ts_orig start time', 'value_1'),
 ('ts_orig end time', 'value_1'),
 ('ts_orig duration', 'value_1'),
 ('dset tr', 'value_1'),
 ('dset start time', 'value_1'),
 ('dset end time', 'value_1'),
 ('dset duration', 'value_1'),
 ('peak num total', 'value_1'),
 ('peak ival min max', 'value_1'),
 (nan, 'value_2'),
 ('peak ival mean std', 'value_1'),
 (nan, 'value_2'),
 ('peak ival q25 q50 q75', 'value_1'),
 (nan, 'value_2'),
 (nan, 'value_3'),
 ('peak num over dset', 'value_1'),
 ('peak ival over dset min max', 'value_1'),
 (nan, 'value_2'),
 ('peak ival over dset mean std', 'value_1'),
 (nan, 'value_2'),
 ('peak

In [40]:
report_card_summary_df = read_gen_ss_review_table(report_card_summary_path)

ValueError: Length mismatch: Expected axis has 35 elements, new values have 27 elements

In [39]:
def read_gen_ss_review_table(file_path):
    """
    Reads the output of AFNI command gen_ss_review_table and
    organizes the contents into a pandas dataframe with meaningful
    column names
    
    Input
    -----
    file_path: path to input file
    
    Returns
    -------
    df: dataframe with the information in the file
    """
    if not osp.exists(file_path):
        print('++ERROR [read_gen_ss_review_table]: input file does not exists')
        return None
    df = pd.read_csv(file_path, sep='\t', header=None)
    original_columns  = [(a,b) for a,b in df.loc[[0,1]].values.T]
    df.drop([0,1], inplace=True)
    new_columns = []
    for (a,b) in original_columns:
        if a=='infile':
            new_columns = new_columns + ['infile']
            continue
        if (a=='echo times'):
            new_columns = new_columns + ['e01','e02','e03']
            continue
        if (a is np.nan):
            continue
        if (a=='orig voxel counts'):
            new_columns = new_columns + ['Nx','Ny','Nz']
            continue
        if (a=='orig voxel resolution'):
            new_columns = new_columns + ['orig Dx','orig Dy','orig Dz']
            continue
        if (a=='final voxel resolution'):
            new_columns = new_columns + ['final Dx','final Dy','final Dz']
            continue
        if (a=='orig volume center'):
            new_columns = new_columns + ['orig volume center x','orig volume center y','orig volume center z']
            continue
        new_columns = new_columns + [a]
    df.columns = new_columns
    for c in df.columns:
        try:
            df[c] = df[c].astype(float)
        except:
            df[c] = df[c]
    print("++ INFO [read_gen_ss_review_table]: Number of scans = %d | Number of metrics per scan = %d" % (df.shape))
    return df


## Check Quality of Physio

In [84]:
df = pd.DataFrame(index=scans_with_complete_physio,columns=['Resp Sampling Freq','Card Sampling Freq','Resp Peak IV Min','Resp Peak IV Max','Resp Peak IV Mean'])
for sbj,ses in scans_with_complete_physio:
    for ec in [2]:
        resp_review_path = osp.join(PRCS_DATA_DIR,sbj,'D06_Physio',f'{sbj}_{ses}_task-rest_echo-{ec}_resp_review.txt')
        resp_review = pd.read_csv(resp_review_path, sep=':', header=None, index_col=0, skipinitialspace=True)
        df.loc[(sbj,ses),'Resp Sampling Freq'] = np.float16(resp_review.loc['read_in sampling freq               '].values[0])
        df.loc[(sbj,ses),'Resp Peak IV Min']   = np.float16(resp_review.loc['peak ival min max                   '].values[0].split(' ')[0])
        df.loc[(sbj,ses),'Resp Peak IV Max']   = np.float16(resp_review.loc['peak ival min max                   '].values[0].split(' ')[1])
        df.loc[(sbj,ses),'Resp Peak IV Mean']  = np.float16(resp_review.loc['peak ival mean std                  '].values[0].split(' ')[0])
df = df.infer_objects()

In [85]:
df[['Resp Peak IV Min','Resp Peak IV Max','Resp Peak IV Mean']].hvplot(hover_cols=['Subject','Session'],width=1500)

:NdOverlay   [Variable]
   :Curve   [Subject]   (value,Session)

In [81]:
resp_review

,1
0,
filename,/data/SFIMJGC_HCP7T/BCBL2024/openeuro/des00359...
label,resp
read_in sampling freq,40
prefilt mode,none
"prefilt window, sec",NaN
is user interact on?,False
num peaks changed,0
num troughs changed,0
ts_orig sampling freq,40.0
